# 03 Edge Training (T4 Schedule)
Notebook-first ShiroNet edge training workflow with adversarial epsilon schedule and versioned artifacts.

## Goal
- Train `shufflenet_v2_x0_5` for edge efficiency
- Use FGSM schedule from `0.002` to `0.012`
- Save checkpoint + metrics for reproducibility

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
print('root:', ROOT)
print('data exists:', (ROOT / 'data').exists())

In [ ]:
!python -m src.data.prepare_dataset --input-root data/raw --output-root data/processed/intel_scenes --val-ratio 0.1

In [ ]:
!python -m src.train --data-root data/processed/intel_scenes --epochs 12 --batch-size 32 --img-size 160 --lr 5e-4 --adv-eps-start 0.002 --adv-eps-end 0.012 --save-dir models/intel_run --pretrained --arch shufflenet_v2_x0_5 --augment-profile shironet

In [ ]:
import pandas as pd
from pathlib import Path

m = Path('models/intel_run/train_metrics.jsonl')
if m.exists():
    df = pd.read_json(m, lines=True)
    display(df.tail())
else:
    print('metrics not found yet')

In [ ]:
HF_REPO = 'ShiroOnigami23/shironet-edge'
!python scripts/hf_upload.py --repo-id {HF_REPO} --local-dir models/intel_run --path-in-repo checkpoints/intel-run-shufflenet-schedule --private